In [1]:
import torch
import torch.nn as nn
import torch.functional as F
import torch.optim as optim

In [7]:
sentences = [
    # --- Category 0: ❤️ (Love, Happy, Positive) ---
    "i love deep learning so much",
    "this neural network model is amazing",
    "transformers are incredible pieces of architecture",
    "i am having so much fun training these networks",
    "python and pytorch make coding beautiful",
    "the community is so supportive and helpful",
    "natural language processing is my favorite subject",
    "i deeply appreciate your clear explanations",
    "learning artificial intelligence feels wonderful",
    "this is an absolutely fantastic day",

    # --- Category 1: 😭 (Sad, Frustrated, Struggling) ---
    "my model is exploding gradients again",
    "i hate debugging these silent runtime errors",
    "so many annoying errors in my code execution",
    "the loss is fluctuating wildly and not converging",
    "cuda went out of memory and crashed my system",
    "i am totally stuck on this bug for hours",
    "nothing is working out today with my python script",
    "training takes forever and the results are terrible",
    "i feel completely defeated by this compiler error",
    "why is my neural network underperforming so badly",

    # --- Category 2: 🎉 (Excited, Celebrating, Success) ---
    "the code finally runs without a single error",
    "we are going to train on a fast t4 gpu",
    "hooray my loss function is decreasing rapidly",
    "i finally passed my machine learning assignment",
    "my validation accuracy hit an all time high score",
    "wow look at this amazing text generation output",
    "i deployment my first real model to production today",
    "yes everything is finally working perfectly now",
    "celebrating because my script finished training early",
    "we successfully built an emoji predictor from scratch"
]

In [16]:
targets = torch.zeros((10,),dtype=torch.long)
targets = torch.cat((targets,targets+1,targets+2),dim=0)
targets

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2,
        2, 2, 2, 2, 2, 2])

In [17]:
emojis = { 0:"❤️",1: "😭", 2: "🎉"}

In [80]:
words = list(set(" ".join(sentences).split()))
word_idx = {word:i+1 for i,word in enumerate(words)}
word_idx['.'] = 0
word_idx = dict(sorted(word_idx.items()))
word_idx
vocab_size = len(word_idx)

In [48]:
max_len = max(len(s.split()) for s in sentences)
max_len

10

In [49]:
def encode(sentence):
    tokens = sentence.split()
    encoded = [word_idx[w] for w in tokens if w in word_idx]
    padded = encoded + [0]*(max_len-len(encoded))
    return padded
print(sentences[1])
encode(sentences[1])

this neural network model is amazing


[102, 132, 32, 54, 57, 20, 0, 0, 0, 0]

In [65]:
X = torch.tensor([encode(s) for s in sentences],dtype=torch.long)
Y = targets.detach().clone()
print(X.shape,Y.shape)

torch.Size([30, 10]) torch.Size([30])


<img src="LSTM_math.png">

### 🧠 Complete Dimension Guide for the Custom LSTM Network

When building a sequence model like an Emoji Predictor, "input" and "output" change meanings depending on whether you look at a single word step under a microscope, or look at the whole sentence model from a bird's-eye view.

---

### 1. Inside the LSTM Cell (The Microscope View)

The `ManualLSTMCell` handles data word-by-word inside a time-step loop. At this level:

* **`input_dim` (Size of one Word Vector):** This is the number of numeric features representing a single word at a single point in time. 
    * *In our code:* It is our `embedding_dim` (`32`).
    * *In the diagram:* It is the width of the grey **Input** wire at the bottom. Every time the loop ticks, a packet of 32 numbers enters the cell.
* **`hidden_dim` (Size of the Memory State):** This is the storage container size for the cell's short-term and long-term memory.
    * *In our code:* It is our `hidden_dim` (`64`).
    * *In the diagram:* It is the width of the maroon arrow exiting the bottom right. The physical size of this memory vector remains completely static (e.g., 64 items) from the first word to the last word; only the *values inside the vector* update as text is processed.

---

### 2. In the Complete Model (The Bird's-Eye View)

The parent `CustomLSTMModel` wraps around the cell to ingest full text strings and output final emoji classifications. At this macro level:

* **`vocab_size` (The Model's Text Input Scope):** The boundary of your overall model is determined by how many unique words are in your dictionary.
    * *In our code:* `vocab_size` (~`65` unique words). It tells the `nn.Embedding` lookup table how many rows it needs to manage.
* **`output_dim` (The Final Prediction Scale):** The number of choices your model has to pick from.
    * *In our code:* `output_dim` (`3` for our target emojis: ❤️, 😭, 🎉).

---

### 3. Complete Data Flow Pipeline

Here is exactly how these dimensions pass the baton to one another as data flows through your custom model:

```text
       DATA TYPE              LAYER MACHINE               VECTOR SHAPE / DIMENSION
       
     [ Raw Text ] 
          │
          ▼
     "deep learning"      ==> [ Tokenizer Mapping ]   ==>  List of Integer IDs
          │
          ▼
   [ Integer IDs ]        ==> [ nn.Embedding ]        ==>  Uses VOCAB_SIZE (65 rows)
          │                                                Outputs EMBEDDING_DIM (32 columns)
          │
          ▼
 ┌────────────────────────────────────────────────────────────────────────────────────────┐
 │                              THE LSTM TIME LOOP LAYER                                  │
 │                                                                                        │
 │   [ Word Vector ]      ==> [ ManualLSTMCell ]      ==>  Takes INPUT_DIM (32)           │
 │                                                         Tracks & Outputs HIDDEN_DIM (64)│
 └────────────────────────────────────────────────────────────────────────────────────────┘
          │
          ▼
  [ Final Memory ]        ==> [ nn.Linear (FC) ]      ==>  Takes HIDDEN_DIM (64)
  Vector of 64 numbers                                     Outputs OUTPUT_DIM (3 emoji scores)
          │
          ▼
     [ Logits ]           ==> [ Argmax / Softmax ]    ==>  Predicts: Class 2 (🎉)

In [90]:
torch.manual_seed(42)
class lstmcell(nn.Module):
    def __init__(self,in_dim,hidden_dim):
        super().__init__()
        self.in_dim = in_dim # the size of number of inputs 
        self.hidden_dim = hidden_dim # the size of number of numbers in short term memmory
        self.gates = nn.Linear(in_dim + hidden_dim, 4 * hidden_dim)
    def forward(self,x,states):
        h_prev,c_prev = states # h_prev = short mem, c_prev = long mem
        combined = torch.cat((x,h_prev),dim=1)
        gate_out = self.gates(combined)
        # Chunk the massive tensor into 4 equal gate splits
        f_gate, i_gate, c_cand, o_gate = gate_out.chunk(4, dim=1)
        # Compute activations matching the diagram's math curves
        f_gate = torch.sigmoid(f_gate) # 🟦 Blue Box (Forget)
        i_gate = torch.sigmoid(i_gate) # 🟩 Green Box (Input)
        c_cand = torch.tanh(c_cand)    # 🟨 Yellow Box (Candidate)
        o_gate = torch.sigmoid(o_gate) # 🟪 Purple Box (Output)
        c_next = (f_gate*c_prev) + (i_gate*c_cand)
        h_next = o_gate*torch.tanh(c_next)
        return (h_next,c_next) # return a tuple
class custom(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_dim,out_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim)
        self.cell = lstmcell(embedding_dim,hidden_dim)
        self.out = nn.Linear(hidden_dim,out_dim)
        self.hidden_dim = hidden_dim
    def forward(self,x):
        embeds = self.embedding(x)
        batch_size, seq_len, _ = embeds.shape
        h = torch.zeros(batch_size,self.hidden_dim)
        c = torch.zeros(batch_size,self.hidden_dim)
        for t in range(seq_len):
            h,c = self.cell(embeds[:,t,:],(h,c))
        logits = self.out(c)
        return logits

In [97]:
EMBEDDING_DIM = 16 # This is our input_dim for the internal cell math
HIDDEN_DIM = 64
OUTPUT_DIM = 3     # 3 emoji categories

In [98]:
model = custom(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM)
optimizer = optim.Adam(model.parameters(), lr=0.005)
criterion = nn.CrossEntropyLoss()
for epoch in range(120):
    logits = model(X)
    loss = criterion(logits, Y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:03d}/120 | Training Loss: {loss.item():.4f}")

Epoch 020/120 | Training Loss: 0.0315
Epoch 040/120 | Training Loss: 0.0000
Epoch 060/120 | Training Loss: 0.0000
Epoch 080/120 | Training Loss: 0.0000
Epoch 100/120 | Training Loss: 0.0000
Epoch 120/120 | Training Loss: 0.0000


In [99]:
model.eval()
with torch.no_grad():
    # Write any test string using vocabulary words to evaluate the hidden state's response
    test_phrase = "we are going to train on a fast t4 gpu"
    
    test_encoded = [word_idx[w] for w in test_phrase.split() if w in word_idx]
    test_padded = test_encoded + [0] * (max_len - len(test_encoded))
    test_tensor = torch.tensor([test_padded], dtype=torch.long)
    
    predictions = model(test_tensor)
    predicted_label = torch.argmax(predictions, dim=-1).item()
    
    print("\n--- Model Inference Test ---")
    print(f"Input Text: '{test_phrase}'")
    print(f"Predicted Sentiment Emoji: {emojis[predicted_label]}")


--- Model Inference Test ---
Input Text: 'we are going to train on a fast t4 gpu'
Predicted Sentiment Emoji: 🎉
